In [1]:
import numpy as np
import pandas as pd

# For statistical tests
from scipy import stats

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# For text EDA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LinearRegression, LogisticRegression

# For entropy computation
from pyitlib import discrete_random_variable as drv

In [2]:
### Load data

train_raw = pd.read_csv("train.csv", header = 0)
test_raw = pd.read_csv("test.csv", header = 0)
edges_raw = pd.read_csv("txs_edgelist.csv", header = 0)
sample_submission = pd.read_csv("sample_submission.csv", header = 0)

In [3]:
### Preview Data

print("train data:\n",train_raw.head(10), "\n")

print("test data:\n", test_raw.head(10), "\n")

# From Kaggle Page, edges is optional to use
print("edges:\n", edges_raw.head(10), "\n")

train data:
         txId  time_step    feat_1    feat_2    feat_3     feat_4    feat_5  \
0  230425980          1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458          1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460          1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397          1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314          1  1.011523 -0.081127 -1.201369   1.153668  0.333276   
5  230459870          1  0.961040 -0.081127 -1.201369   1.303743  0.333276   
6  230333930          1 -0.171264 -0.184668 -1.201369  -0.121970 -0.043875   
7  230595899          1 -0.171755 -0.184668 -1.201369  -0.046932 -0.043875   
8  232013274          1 -0.123127 -0.184668 -1.201369  -0.121970 -0.043875   
9  232029206          1 -0.005027  0.578941 -0.091383   4.380281 -0.063725   

     feat_6     feat_7    feat_8  ...  feat_157  feat_158  feat_159  feat_160  \
0 -0.113002  -0.061584 -0.162097  ... -0.600999

## 2.1 Data Preprocessing and Feature Engineering

### Data Cleaning and Transformation
* **Data Cleaning & Preparation:** Describe how the data was cleaned, transformed, and prepared.
* **Handling Missing Values:** Detail decisions and methods used for handling missing values, justified by domain insights or experimental evidence.


In [ ]:
### Edgelist - is optional use, leave to the back burner for now

print("check for self-payments:", len(edges[edges["txId1"] == edges["txId2"]]))

check = edges.groupby(by = ["txId1", "txId2"], as_index = False).size()

print(check)

check for self-payments: 0
234354    1
0         1
1         1
234338    1
234337    1
         ..
7         1
6         1
5         1
4         1
3         1
Name: size, Length: 234355, dtype: int64


In [8]:
### Train/Test:
### Assessment of NA values - label

'''
feature_1 … feature_165 — anonymized, pre-normalized numeric features describing each transaction. These fall into two broad groups, though the exact identity of each column is not disclosed:
Local features: information about the transaction itself (e.g. transaction volume, fee, number of inputs/outputs).
Aggregated features: statistics (max, min, standard deviation, correlation, etc.) computed over the transaction's immediate neighbors in the graph.
'''

num_label_NA = len(train_raw[train_raw["label"].isna()])

num_label_total = len(train_raw)

print("total number of NA:", num_label_NA)
print("total training data:", num_label_total)

print("portion of NA:", (num_label_NA * 100/num_label_total))

print("unique label values:" ,train_raw["label"].unique())


total number of NA: 110537
total training data: 141772
portion of NA: 77.96814603729932
unique label values: [nan  0.  1.]


In [7]:
# for now, drop NA labels
train_labeled = train_raw[train_raw["label"].isin([0, 1, 0.0, 1.0])].copy()
train_labeled["label"] = train_labeled["label"].astype(int)

print(f"Total train rows: {len(train_raw)}")
print(f"Usable labeled rows for training: {len(train_labeled)}")

Total train rows: 141772
Usable labeled rows for training: 31235


### Feature Engineering
* **Graph-Based Features:** 
  * Node degree
  * Neighborhood aggregates
  * Other engineered graph features
* **Justification:** Justify the graph-based choices using domain insights or experimental evidence.

---

### Feature Selection, Extraction, and Augmentation
* **Feature Selection & Extraction:** Outline the methods and criteria applied for selecting and extracting features.
* **Augmentation Techniques:** Detail any data augmentation techniques used.
* **Justification:** Provide domain insights or experimental evidence justifying these decisions.